# 🏠 01 — Exploración: Idealista API
Fuente: Idealista Property Search API v3.5  
Datos: precios de venta y alquiler en las 7 principales ciudades españolas  
Objetivo: entender la estructura del JSON, distribución de precios y cobertura geográfica

Celda 2 — Imports y carga del JSON más reciente

In [1]:
import json
import pandas as pd
import glob
import os
from pathlib import Path

ROOT = Path(os.getcwd()).parent.parent  # sube desde notebooks/01_exploracion/ a raíz
DATA_DIR = ROOT / "data" / "idealista"

# Carga el JSON más reciente disponible
archivos = sorted(glob.glob(str(DATA_DIR / "**" / "*.json"), recursive=True))
if not archivos:
    raise FileNotFoundError(f"No hay datos de Idealista en {DATA_DIR}. Ejecuta trigger.py primero.")

ultimo = archivos[-1]
print(f"📂 Cargando: {ultimo}")
with open(ultimo) as f:
    raw = json.load(f)

print(f"✅ Snapshot del: {raw.get('snapshot_date', 'desconocido')}")
print(f"📊 Peticiones usadas ese mes: {raw.get('peticiones_usadas_este_mes', '?')}/100")

FileNotFoundError: No hay datos de Idealista en /workspaces/TFG_Spain-s_Migratory_Flow/data/idealista. Ejecuta trigger.py primero.

Celda 3 — Convertir a DataFrame

In [ ]:
registros = raw.get("anuncios", [])
df = pd.DataFrame(registros)

print(f"Total anuncios: {len(df)}")
print(f"Columnas: {list(df.columns)}")
df.head(3)

Celda 4 — Info básica

In [ ]:
df.info()

Celda 5 — Distribución por ciudad y operación

In [ ]:
resumen = df.groupby(["city", "operation"]).agg(
    n_anuncios     = ("propertyCode", "count"),
    precio_medio   = ("price",    "mean"),
    precio_m2_medio= ("price_m2", "mean"),
    precio_m2_mediana=("price_m2","median"),
).round(2)

print(resumen.to_string())

Celda 6 — Nulos y cobertura

In [ ]:
nulos = df.isnull().sum()
pct   = (nulos / len(df) * 100).round(1)
pd.DataFrame({"nulos": nulos, "pct_%": pct}).query("nulos > 0").sort_values("pct_%", ascending=False)

Celda 7 — Estadísticas de precio/m²

In [ ]:
df.groupby("operation")["price_m2"].describe().round(2)

Celda 8 — Resumen del snapshot por ciudad (del JSON original)

In [ ]:
for ciudad, ops in raw.get("ciudades", {}).items():
    for op, stats in ops.items():
        if isinstance(stats, dict) and "error" not in stats:
            print(f"  {ciudad:12} {op:5} → mercado total: {stats.get('total_anuncios_mercado'):>6} | "
                  f"capturados: {stats.get('anuncios_capturados'):>3} | "
                  f"precio medio: {stats.get('precio_medio_m2'):>7} €/m²")